# BEATs mechanical-confuser comparison

This continues the **complete** previous public-corpus study, using its existing
BEATs embeddings and trained head. It compares the original detector, a broader
decision threshold, and a head trained with additional mechanical negatives.
The pretrained BEATs encoder remains frozen.

The previous test is consumed development data. The new public challenge uses
all eligible FSD50K evaluation recordings, supported WSU flight/background
intervals, and indexed AuDroK point recordings. Publisher labels are weak;
three-second crops are not independent field trials or false-alarm/hour exposure.

Use the same CPU/GPU setting as the previous full study. Its complete BEATs
checkpoint, frozen head, and all222 original archive receipts must exist. This
notebook verifies those inputs before downloading the additional6.26GB FSD
audio. It processes the complete reserved cohort and does not substitute a
small demonstration when inputs are missing. New results and model parameters
persist in a separate Drive project. A published evaluation cohort is consumed;
rerunning it is a reproduction, not another fresh generalization test.


In [ ]:
ARCHIVES_DRIVE_PATH = '/content/drive/MyDrive/drone-detection-v2/corpus-intake-v3/source-archives'
PREVIOUS_PROJECT_DRIVE_PATH = '/content/drive/MyDrive/drone-corpus-study-v1-20261115'
PROJECT_DRIVE_PATH = '/content/drive/MyDrive/drone-beats-confusers-v1-20261116'
REPO_URL = 'https://github.com/StoneDerek/drone-detection.git'
CODE_COMMIT = '934dd24389191efd265862c8f1ae94946a380231'
DEVICE = 'cuda'  # Must match the previous complete study's BEATs encoder identity.
FEATURE_SHARDS = 1


In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import urllib.request
from pathlib import Path

from google.colab import drive

if DEVICE not in {'cpu', 'cuda'} or FEATURE_SHARDS not in {1, 2, 3, 4}:
    raise ValueError('Choose an explicit supported device and worker count')
if DEVICE == 'cuda':
    subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'], check=True)
drive.mount('/content/drive')
MOUNT = Path('/content/drive').resolve()
if not os.path.ismount(MOUNT):
    raise RuntimeError('Drive must be mounted for durable results')
PROJECT = Path(PROJECT_DRIVE_PATH).resolve()
ARCHIVES = Path(ARCHIVES_DRIVE_PATH).resolve()
if not PROJECT.is_relative_to(MOUNT) or PROJECT == MOUNT or not ARCHIVES.is_relative_to(MOUNT):
    raise ValueError('Choose the expected mounted Drive paths')
if PROJECT == ARCHIVES or PROJECT.is_relative_to(ARCHIVES) or ARCHIVES.is_relative_to(PROJECT):
    raise ValueError('Keep the result project separate from the input archives')
PROJECT.mkdir(parents=True, exist_ok=True)
if (PROJECT / 'bootstrap.json').exists() and json.loads((PROJECT / 'bootstrap.json').read_text()).get('revision') != CODE_COMMIT:
    raise ValueError('This project has another code pin; preserve it and use a new project folder')
print('Input archives:', ARCHIVES)
print('Durable results:', PROJECT)
print('Local runtime disk:', shutil.disk_usage('/content'))


In [ ]:
"""Standard-library-only bootstrap, embedded verbatim in the Colab notebook."""

import hashlib
import json
import os
import re
import subprocess
import uuid
from pathlib import Path


def git_output(checkout, *arguments):
    return subprocess.check_output(['git', '-C', str(checkout), *arguments], text=True).strip()


def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def pin_checkout(project, checkout, repo_url, first_ref='main'):
    """Resolve once, then reuse that exact revision after runtime replacement.

    One active runtime per project. Refuse dirty or unrelated local checkouts.
    The saved lock hash is also checked; a pin is never silently refreshed.
    """
    project, checkout = Path(project), Path(checkout)
    project.mkdir(parents=True, exist_ok=True)
    pin_path = project / 'bootstrap.json'
    pin = json.loads(pin_path.read_text()) if pin_path.exists() else None
    if pin is not None and (pin.get('schema_version') != 1
                            or pin.get('repo_url') != repo_url
                            or not re.fullmatch(r'[0-9a-f]{40}', pin.get('revision', ''))
                            or not re.fullmatch(r'[0-9a-f]{64}', pin.get('runtime_lock_sha256', ''))):
        raise ValueError('Invalid/different project pin; choose a new project folder deliberately')
    if checkout.exists() and any(checkout.iterdir()):
        if not (checkout / '.git').exists():
            raise ValueError('Checkout folder contains unrelated files')
        if git_output(checkout, 'remote', 'get-url', 'origin') != repo_url:
            raise ValueError('Checkout has a different repository origin')
        if git_output(checkout, 'status', '--porcelain'):
            raise ValueError('Local checkout changed; preserve those changes before continuing')
    else:
        checkout.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', 'init', '--quiet', str(checkout)], check=True)
        subprocess.run(['git', '-C', str(checkout), 'remote', 'add', 'origin', repo_url], check=True)
    target = pin['revision'] if pin else first_ref
    subprocess.run(['git', '-C', str(checkout), 'fetch', '--quiet', '--depth', '1', 'origin', target], check=True)
    revision = git_output(checkout, 'rev-parse', 'FETCH_HEAD')
    if pin and revision != pin['revision']:
        raise ValueError('Fetched revision differs from saved project pin')
    subprocess.run(['git', '-C', str(checkout), 'checkout', '--quiet', '--detach', revision], check=True)
    candidate = {'schema_version': 1, 'repo_url': repo_url, 'revision': revision,
                 'runtime_lock_sha256': file_hash(checkout / 'notebooks/runtime/uv.lock')}
    if pin is not None and pin != candidate:
        raise ValueError('Pinned environment lock changed')
    if pin is None:
        payload = json.dumps(candidate, indent=2, sort_keys=True).encode()
        temporary = pin_path.with_name('bootstrap.partial-' + uuid.uuid4().hex)
        temporary.write_bytes(payload)
        if pin_path.exists() and json.loads(pin_path.read_text()) != candidate:
            raise ValueError('Another bootstrap wrote a different pin; use one active runtime')
        os.replace(temporary, pin_path)
        if pin_path.read_bytes() != payload:
            raise ValueError('Project pin read-back failed')
    return candidate


REPO = Path('/content/drone-corpus-code')
PIN = pin_checkout(PROJECT, REPO, REPO_URL, CODE_COMMIT)
assert PIN['revision'] == CODE_COMMIT
print('Exact implementation:', PIN['revision'])
print('Runtime lock SHA-256:', PIN['runtime_lock_sha256'])


In [ ]:
UV_DIR = Path('/content/drone-corpus-bootstrap')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', '--target', str(UV_DIR), 'uv==0.13.5'], check=True)
UV_ENV = dict(os.environ, PYTHONPATH=str(UV_DIR), UV_CACHE_DIR='/content/drone-corpus-uv-cache', UV_PYTHON_INSTALL_DIR='/content/drone-corpus-python')
subprocess.run([sys.executable, '-m', 'uv', 'sync', '--project', str(REPO / 'notebooks/runtime'), '--frozen', '--python', '3.12',
                '--extra', 'gpu' if DEVICE == 'cuda' else 'cpu', '--extra', 'probe'], env=UV_ENV, check=True)
PYTHON = REPO / 'notebooks/runtime/.venv/bin/python'
JOB_ENV = dict(os.environ, PYTHONPATH=str(REPO / 'src'), OPENBLAS_NUM_THREADS='1', OMP_NUM_THREADS='1', CUBLAS_WORKSPACE_CONFIG=':4096:8')
check = "import torch; print('Torch:',torch.__version__); "
check += "assert torch.cuda.is_available(); print('GPU:',torch.cuda.get_device_name(0))" if DEVICE == 'cuda' else "print('Explicit CPU execution; full corpus will still be processed')"
subprocess.run([str(PYTHON), '-c', check], cwd=REPO, env=JOB_ENV, check=True)

def run_script(name, *arguments):
    command = [str(PYTHON), str(REPO / 'scripts' / name), *[str(value) for value in arguments]]
    subprocess.run(command, cwd=REPO, env=JOB_ENV, check=True)


In [ ]:
PREVIOUS_PROJECT = Path(PREVIOUS_PROJECT_DRIVE_PATH).resolve()
if not PREVIOUS_PROJECT.is_relative_to(MOUNT) or PREVIOUS_PROJECT == PROJECT:
    raise ValueError('Choose separate mounted Drive projects for previous and new results')
for path in [PREVIOUS_PROJECT / DEVICE / 'models/beats/model.json',
             PREVIOUS_PROJECT / DEVICE / 'models/beats/head.npz',
             PREVIOUS_PROJECT / DEVICE / 'feature-checkpoints/beats.tar.gz']:
    if not path.exists():
        raise RuntimeError('The previous full BEATs study is required: ' + str(path))
STUDY = Path('/content/drone-confuser-previous-' + DEVICE)
identity = {'project': str(PROJECT), 'archives': str(ARCHIVES), 'code': CODE_COMMIT, 'device': DEVICE}
identity_path = STUDY / 'local-study.json'
if STUDY.exists() and any(STUDY.iterdir()) and (not identity_path.exists() or json.loads(identity_path.read_text()) != identity):
    raise ValueError('Preserve the existing local study before starting an unrelated one')
STUDY.mkdir(parents=True, exist_ok=True)
identity_path.write_text(json.dumps(identity, sort_keys=True, indent=2))
MODEL_RESULTS = Path(PREVIOUS_PROJECT_DRIVE_PATH).resolve() / DEVICE / 'models'
MODEL_RESULTS.mkdir(parents=True, exist_ok=True)
for name, target in [('archives', ARCHIVES), ('models', MODEL_RESULTS)]:
    link = STUDY / name
    if not link.exists():
        link.symlink_to(target, target_is_directory=True)
    if link.resolve() != target:
        raise ValueError('Unexpected local study link: ' + name)
intake = json.loads((REPO / 'data/corpus-intake-v3.json').read_text())
missing = []
for item in intake['items']:
    source = ARCHIVES / item['source_id'] / item['filename']
    complete = source.with_name(source.name + '.complete.json')
    if not source.exists() or not complete.exists():
        missing.append(item['source_id'] + '/' + item['filename'])
        continue
    receipt = json.loads(complete.read_text())
    if source.stat().st_size != item['bytes'] or receipt.get('md5') != item['md5'] or not receipt.get('publisher_checksum_verified'):
        raise ValueError('Unverified source: ' + str(source))
if missing:
    raise RuntimeError('Finish the full intake before this study; missing distributions: ' + repr(missing))
print('Complete source inventory:', len(intake['items']), 'files;', sum(i['bytes'] for i in intake['items']), 'bytes')

URBAN = STUDY / 'urbansound.csv'
if not URBAN.exists():
    url = 'https://raw.githubusercontent.com/MOULIK-RAZDAN/AUDIO-CLASSIFICATION/main/UrbanSound8K.csv'
    with urllib.request.urlopen(url, timeout=60) as response:
        URBAN.write_bytes(response.read())
assert hashlib.sha256(URBAN.read_bytes()).hexdigest() == 'b207f7a53f9e5eb02a5dd71d58fb8b66c88c24c0637b926dc72d9d400a6a2354'
PREPARED = STUDY / 'prepared'
run_script('prepare_corpus_study.py', '--archives', ARCHIVES, '--output', PREPARED, '--urban-csv', URBAN)
run_script('finalize_corpus_study.py', '--prepared', PREPARED, '--archives', ARCHIVES)
cohort = json.loads((PREPARED / 'preparation-complete.json').read_text())
print(json.dumps(cohort, indent=2))


In [ ]:
PREVIOUS = STUDY
run_script('checkpoint_corpus_features.py', 'restore', '--directory', PREVIOUS / 'features/beats',
           '--checkpoint', PREVIOUS_PROJECT / DEVICE / 'feature-checkpoints/beats.tar.gz')
old_model = json.loads((PREVIOUS / 'models/beats/model.json').read_text())
cached_encoder = json.loads((PREVIOUS / 'features/beats/encoder.json').read_text())
if old_model['encoder'] != cached_encoder or cached_encoder['device'] != DEVICE:
    raise ValueError('Previous model, cached embeddings and chosen execution device must agree')
ASSETS = PROJECT / 'model-assets/beats'
run_script('fetch_beats.py', '--output', ASSETS)
WORK = PROJECT / DEVICE / 'confuser-study-v1'
WORK.mkdir(parents=True, exist_ok=True)
run_script('acquire_confuser_study.py', '--work', WORK, '--metadata-only')
run_script('prepare_confuser_study.py', 'plan', '--work', WORK, '--previous', PREVIOUS, '--urban-metadata', URBAN)
if not (WORK / 'models/frozen-models.json').exists():
    run_script('run_confuser_study.py', 'fit', '--work', WORK, '--previous', PREVIOUS)
print('All three variants frozen before fresh audio scoring:', WORK / 'models/frozen-models.json')


In [ ]:
run_script('acquire_confuser_study.py', '--work', WORK)
for stage in ['wsu', 'audrok', 'fsd', 'seal']:
    run_script('prepare_confuser_study.py', stage, '--work', WORK)
run_script('run_confuser_study.py', 'authorize', '--work', WORK)
FRESH_FEATURES = WORK / 'features/beats'
CHECKPOINT = PROJECT / DEVICE / 'feature-checkpoints/beats.tar.gz'
run_script('checkpoint_corpus_features.py', 'restore', '--directory', FRESH_FEATURES, '--checkpoint', CHECKPOINT)
run_script('extract_corpus_study_features.py', '--prepared', WORK / 'prepared', '--output', FRESH_FEATURES,
           '--family', 'beats', '--assets', ASSETS, '--device', DEVICE, '--threads', '1', '--splits', 'final')
run_script('checkpoint_corpus_features.py', 'pack', '--directory', FRESH_FEATURES, '--checkpoint', CHECKPOINT)


In [ ]:
REPORT = PROJECT / DEVICE / 'report'
run_script('run_confuser_study.py', 'evaluate', '--work', WORK, '--report', REPORT)
shutil.copyfile(REPO / 'reports/confuser-study-v1/PROTOCOL.md', REPORT / 'PROTOCOL.md')
run_script('report_confuser_study.py', '--work', WORK, '--report', REPORT)
import pandas as pd
display(pd.read_csv(REPORT / 'metrics.csv'))
print('Comparison:', REPORT / 'comparison.json')
print('Per-crop scores:', WORK / 'models/final-predictions.jsonl')
print('Complete source/quality receipts:', WORK / 'prepared/preparation-complete.json')
print('The cohort is now consumed. Do not tune on it and call the next run fresh evaluation.')
